In [155]:
from doctr.io import DocumentFile
from doctr.models import ocr_predictor

model = ocr_predictor(pretrained=True)

def extract_text(ocr_model, image_file_path: str):
    """Extrae el texto de una imagen a partir de un modelo OCR"""

    doc = DocumentFile.from_images(image_file_path)
    result = ocr_model(doc)
    return result

def show_ocr_result(result):
    """Muestra una imagen de resultado y el texto extraido"""

    # mostrar resultado
    #result.show()
    lista = []
    # mostrar texto con mas de 50% de confianza
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                words = []
                for word in line.words:
                    if word.confidence > 0.5:
                        words.append(word.value)
                lista.append(' '.join(words))

    return lista

In [157]:

img1  = 'dni_miguel.jpeg'
img2 = 'dni_miguel_reverso.jpeg'

img3 = 'dni_xaro.jpeg'    
img4 = 'dni_xaro_reverso.jpeg'

text = extract_text(model, './DNI_especimen/' + img1)
words_ante = show_ocr_result(text)

#print(words_ante)
#for i in range(len(words_ante)):
#    print(words_ante[i])


text = extract_text(model, './DNI_especimen/' + img2)
words_reve = show_ocr_result(text)

#print(words_reve)
#for i in range(len(words_reve)):
#    print(words_reve[i])

In [ ]:
info = {}

for i in range(len(words_ante)):
    if len(words_ante[i]) == 9 and words_ante[i].isalnum():
        if words_ante[i][:8].isdigit() and words_ante[i][8:].isalpha():
            info['dni'] = words_ante[i]
        
        else:
            info['num_sop'] = words_ante[i]

    elif 'APELL' in words_ante[i]:
        info['apellido1'] = words_ante[i+1]

        if words_ante[i+2] != 'NOMBRE/NOM' and words_ante[i+2] != 'NOMBRE':
            info['apellido2'] = words_ante[i+2]

    elif 'NOM' in words_ante[i] :
        if ' ' in words_ante[i+1]:
            info['nombre1'] = words_ante[i+1].split()[0]
            info['nombre2'] = words_ante[i+1].split()[1]   
        else:
            info['nombre'] = words_ante[i+1]

    elif words_ante[i] == 'ESP' and not words_ante[i+1].isalpha():
        info['f_nacimiento'] = words_ante[i+1]
        

    elif words_ante[i] == 'VALIDEZ/VALIDESA' or words_ante[i] == 'VALIDEZ':
        if len(words_ante[i+1]) == 10 and not words_ante[i+1].isalpha():
            #info['f_emision'] = words_ante[i+1]
            info['f_validez'] = words_ante[i+2]

        elif len(words_ante[i+1]) == 9 and not words_ante[i+1].isalpha():
            info['f_validez'] = words_ante[i+1]

        else:
            partes = words_ante[i+1].split()
            
            fecha = []
            fecha.append(partes[3])
            fecha.append(partes[4])
            fecha.append(partes[5])
            info['f_validez'] = ' '.join(fecha)



info

{'dni': '21993594M',
 'num_sop': 'CLK123980',
 'apellido1': 'PEREZ',
 'apellido2': 'ALEDO',
 'nombre1': 'MIGUEL',
 'nombre2': 'ANGEL',
 'f_nacimiento': '13 06 1964',
 'f_validez': '25 09 2035'}

In [ ]:
n = len(words_reve)

fila1 = words_reve[n-3]
fila2 = words_reve[n-2]
fila3 = words_reve[n-1]

info_fila1 = []
info_fila2 = []
info_fila3 = []

info_fila1.append(fila1[:5])
info_fila1.append(fila1[5:14])
info_fila1.append(fila1[14:15])
info_fila1.append(fila1[15:24])
info_fila1.append(fila1[25:])
print(info_fila1)

info_fila2.append(fila2[:6])
info_fila2.append(fila2[6:8])
info_fila2.append(fila2[8:14])
info_fila2.append(fila2[14:15])
info_fila2.append(fila2[15:18])
info_fila2.append(fila2[18:27])
info_fila2.append(fila2[27:])
print(info_fila2)


nom_completo = []
compues = False
for clave, values in info.items():
    if clave == 'apellido1' or clave == 'apellido2':
        nom_completo.append(values)

    elif clave == 'nombre1':
        nom_completo.append(values)
    
    elif clave == 'nombre2':
        nom_completo.append(values)
        compues = True
    

ap1 = len(nom_completo[0])
ap2 = len(nom_completo[1])
n1 = len(nom_completo[2])


info_fila3.append(fila3[:ap1])
info_fila3.append(fila3[ap1:ap1+1])
info_fila3.append(fila3[ap1+1:(ap1+ap2)+1])
info_fila3.append(fila3[(ap1+ap2)+1:(ap1+ap2)+3])
info_fila3.append(fila3[(ap1+ap2)+3:(ap1+ap2)+3+n1])
info_fila3.append(fila3[(ap1+ap2)+3+n1: (ap1+ap2)+3+n1+1])
info_fila3.append(fila3[(ap1+ap2)+3+n1+1:])
print(info_fila3)

['IDESP', 'CLK123980', '0', '21993594M', '<<<<<']
['640613', '2M', '350925', '0', 'ESP', '<<<<<<<<<', '<<4']
['PEREZ', '<', 'ALEDO', '<<', 'MIGUEL', '<', 'ANGEL<<<<<']


In [147]:
valor = info_fila2[0]
grupos = [valor[i:i+2] for i in range(0, len(valor), 2)]
f_nac = ' '.join(grupos[::-1])

valor = info_fila2[2]
grupos = [valor[i:i+2] for i in range(0, len(valor), 2)]
f_val = ' '.join(grupos[::-1])

info_reverso = {
    'num_sop': info_fila1[1],
    'dni': info_fila1[3],
    'f_nacimiento': f_nac,
    'f_validez': f_val,
    'apellido1': info_fila3[0],
    'apellido2': info_fila3[2],
    'nombre1': info_fila3[4],
}

if compues:
    info_reverso['nombre2'] = info_fila3[6] 

info_reverso

{'num_sop': 'CLK123980',
 'dni': '21993594M',
 'f_nacimiento': '13 06 64',
 'f_validez': '25 09 35',
 'apellido1': 'PEREZ',
 'apellido2': 'ALEDO',
 'nombre1': 'MIGUEL',
 'nombre2': 'ANGEL<<<<<'}

In [152]:
for clave1, value1 in info.items():
    for clave2, value2 in info_reverso.items():
        if clave1 == 'dni' and clave2 == 'dni':
            if value1 == value2:
                print('El dni coincide')
            else:
                print('El dni no coincide')
        
        elif clave1 == 'num_sop' and clave2 == 'num_sop':
            if value1 == value2:
                print('El num_sop coincide')
            else:
                print('El num_sop no coincide')
        
        elif clave1 == 'f_nacimiento' and clave2 == 'f_nacimiento':
            aux = value1[:2]+ value1[2:6]+value1[8:]
            if aux == value2:
                print('La fecha de nacimiento coincide')
            else:
                print('La fecha de nacimiento no coincide')
        
        elif clave1 == 'f_validez' and clave2 == 'f_validez':
            aux = value1[:2]+ value1[2:6]+value1[8:]
            if aux == value2:
                print('La fecha de validez coincide')
            else:
                print('La fecha de validez no coincide')
            
        elif clave1 == 'apellido1' and clave2 == 'apellido1':
            if value1 == value2:
                print('El primer apellido coincide')
            else:
                print('El primer apellido no coincide')
        
        elif clave1 == 'apellido2' and clave2 == 'apellido2':
            if value1 == value2:
                print('El segundo apellido coincide')
            else:
                print('El segundo apellido no coincide')

        elif clave1 == 'nombre1' and clave2 == 'nombre1':
            if value1 == value2:
                print('El primer nombre coincide')
            else:
                print('El primer nombre no coincide')

        if clave1 == 'nombre2' and clave2 == 'nombre2':
            if value1 in value2:
                print('El segundo nombre coincide')
            else:
                print('El segundo nombre no coincide')
        
        
    

El dni coincide
El num_sop coincide
El primer apellido coincide
El segundo apellido coincide
El primer nombre coincide
El segundo nombre coincide
La fecha de nacimiento coincide
La fecha de validez coincide
